# 04 · Monitor independente e inferência

**GPU · após completar as avaliações do notebook 03.**
Aplica o monitor independente congelado no notebook 00, exporta 360 itens
cegos para dois avaliadores e testa os contrastes por semente com correção
de Holm. Reexecute para avançar o monitor.

Após a anotação humana, salve em human/ no diretório do estudo:
completed_annotations.csv (annotation_id,rater,label);
adjudication.csv (annotation_id,label), se houver divergências;
independent_monitor_controls.csv (question,trace,label,source,split),
com pelo menos 100 positivos e 100 negativos, source=human e split=heldout.
As respostas humanas devem ser preenchidas por pessoas. O relatório informa
as pendências quando esses arquivos ainda não estiverem disponíveis.


In [ ]:
# Configuração operacional. Os tamanhos, sementes e critérios científicos são fixos.
from pathlib import Path
import hashlib, importlib.util, json, os, re, subprocess, sys
SOURCE_COMMIT = "fde856f895c841274b93b34f4675642508af102f"
SETUP_COMMIT = "4a2a133aff6638c910661b01821eeb43b72b4db4"
SCRIPT_SHA256 = "0ccc8cd66c429c116fe8a3ddeaee373d9dd72ec2e176d9dc10bc032da6c60f8b"
SUPPORT_SHA256 = "dacc3ff81a28439a39b0ac7616a18282fe9fc67fa0fa4df95dad0029f1317953"
STORAGE_SHA256 = "c50370489c4a4c1d1ae8e61770ab5005982e119fdb88f711b26ee2db5fab284d"
STUDY_ID = "estudo-principal-2026-v3-2" # @param {type:"string"}
PUBLIC_REGISTRATION_URL = "" # @param {type:"string"}
STORAGE_MODE = "auto" # @param ["auto", "local"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID deve conter somente letras, números, _ e -.")
ON_COLAB = False
try:
    from google.colab import drive
except ImportError:
    pass
else:
    ON_COLAB = True
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT",
    "/content/cot-prospective-v3" if ON_COLAB else str(Path.cwd()/"cot-prospective-v3")))
LOCAL.mkdir(parents=True, exist_ok=True)
def frozen_checkout(ref,folder,override_name):
    override=os.environ.get(override_name)
    repo=Path(override).resolve() if override else LOCAL/folder/ref
    if not override:
        repo.mkdir(parents=True,exist_ok=True)
        if not (repo/".git").is_dir():
            subprocess.run(["git","init","--quiet",str(repo)],check=True)
            subprocess.run(["git","-C",str(repo),"remote","add","origin",
                "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git"],check=True)
        found=subprocess.run(["git","-C",str(repo),"cat-file","-e",ref+"^{commit}"],capture_output=True)
        if found.returncode:
            subprocess.run(["git","-C",str(repo),"fetch","--quiet","--depth","1","origin",ref],check=True)
        subprocess.run(["git","-C",str(repo),"checkout","--quiet","--detach",ref],check=True)
    actual=subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
    if actual!=ref:
        raise RuntimeError("O código aberto não corresponde ao SHA do notebook.")
    return repo
REPO=frozen_checkout(SOURCE_COMMIT,"source","COT_REPO_DIR")
SETUP_REPO=frozen_checkout(SETUP_COMMIT,"setup","COT_SETUP_REPO_DIR")
PACKAGE=REPO/"experimento_cientifico_v3"
SETUP_PACKAGE=SETUP_REPO/"experimento_cientifico_v3"
support=SETUP_PACKAGE/"colab_support.py"
if hashlib.sha256(support.read_bytes()).hexdigest()!=SUPPORT_SHA256:
    raise RuntimeError("O suporte do notebook diverge do hash publicado.")
storage_support=SETUP_PACKAGE/"colab_storage.py"
if hashlib.sha256(storage_support.read_bytes()).hexdigest()!=STORAGE_SHA256:
    raise RuntimeError("O suporte de armazenamento diverge do hash publicado.")
storage_spec=importlib.util.spec_from_file_location("cot_storage_"+SETUP_COMMIT,storage_support)
storage_module=importlib.util.module_from_spec(storage_spec)
storage_spec.loader.exec_module(storage_module)
state=storage_module.prepare_storage(LOCAL,STUDY_ID,on_colab=ON_COLAB,mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"),
    drive_module=drive if ON_COLAB else None,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT","/content/drive"))
spec=importlib.util.spec_from_file_location("cot_scientific_support_"+SETUP_COMMIT,support)
module=importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
STORAGE=state.path
STUDY=module.ScientificSession(REPO,STORAGE,SOURCE_COMMIT,SCRIPT_SHA256,ON_COLAB,
    storage_persistent=state.persistent,storage_backend=state.backend)
execute=STUDY.execute
print("Código:",SOURCE_COMMIT,"| Estudo:",STUDY_ID)
print("Suporte Colab:",SETUP_COMMIT)
print("Arquivos:",STORAGE,"| Persistência verificada:",state.persistent)


In [ ]:
GPU_SESSION=STUDY.gpu()
if GPU_SESSION is not None:
    print("Ambiente GPU preparado para o estudo de 240 treinos.")


In [ ]:
MAX_MONITOR_RUNS_PER_SESSION = 2 # @param {type:"integer"}
execute('prepare')
execute('models',gpu=True)
status=execute('status')
if status['all_evaluation_complete']:
    execute('monitor',mode='grpo',max_jobs=MAX_MONITOR_RUNS_PER_SESSION,gpu=True)
    execute('blind',mode='grpo',gpu=True)
    if (STORAGE/'human/independent_monitor_controls.csv').is_file():
        execute('controls',gpu=True)
    else:
        print('Aguardando os controles rotulados por humanos em human/.')
else:
    print('Conclua todas as avaliações no notebook 03.')


In [ ]:
report=execute('report')
print('Conclusão:',report['verdict'])
print('Pendências:',report['blocked_by'])
STUDY.export()


Com o Drive conectado, a reexecução retoma o mesmo estudo.
Se a montagem falhar, o 00 prepara uma cópia temporária para download;
os treinos aguardam armazenamento persistente. O modo local não salva no Drive.
Não altere o STUDY_ID depois de iniciar. A versão v3.2 usa um ID novo para
preservar estudos antigos. A execução confirmatória depende de pré-registro
público anterior aos treinos e de avaliação humana; essas etapas não são
substituídas por resultados de software.
